# Speech AI Fundamentals

**Module:** 16 — Speech AI

Map ASR, TTS, audio understanding, and voice agents — plus audio basics and constraints.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Map the speech AI landscape: ASR, TTS, understanding, voice agents
- Explain sample rate, codecs, mono/stereo, streaming vs batch
- Identify latency, quality, privacy, and accessibility constraints
- Generate a tiny WAV and inspect its properties


## Pillars of Speech AI

### Definition
Speech AI covers systems that turn sound into meaning and meaning into sound — often inside a conversational loop.

### Why it matters
Voice is the most natural UI for many users and the harshest real-time environment for LLMs.

### How it works
Four pillars: **ASR** (speech→text), **TTS** (text→speech), **audio understanding** (intent/emotion/events/diarization), **voice agents** (full conversational systems).

### Intuition
Ears, mouth, brain, and turn-taking — missing any one feels broken.

### Pitfalls
- Building only ASR+TTS and calling it an agent
- Ignoring telephony codecs (8 kHz) vs app audio (16/48 kHz)

### When to use
Contact centers, accessibility, meeting intelligence, voice UX in apps/cars.


### Landscape map

| Pillar | Input | Output | Key metric |
|--------|-------|--------|------------|
| ASR | Audio | Text (+timestamps) | WER / latency |
| TTS | Text/SSML | Audio | MOS / time-to-first-byte |
| Understanding | Audio/text | Labels/structure | F1 / diarization error |
| Voice agents | Duplex audio | Actions+speech | Task success / barge-in UX |

```mermaid
flowchart LR
  M[Mic / PSTN] --> A[ASR]
  A --> U[NLU / LLM]
  U --> T[TTS]
  T --> S[Speaker]
  M --> D[Diarization / events]
  D --> U
```


In [ ]:
# Demo 1: write a sine WAV and inspect headers
import wave, struct, math
from pathlib import Path

def write_sine_wav(path, seconds=0.3, freq=440.0, rate=16000):
    n = int(seconds * rate)
    with wave.open(path, "w") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
        frames = bytearray()
        for i in range(n):
            val = int(3000 * math.sin(2 * math.pi * freq * (i / rate)))
            frames.extend(struct.pack("<h", val))
        w.writeframes(frames)
    return path

out = write_sine_wav("demo_tone.wav")
with wave.open(out) as r:
    print({"path": out, "rate": r.getframerate(), "ch": r.getnchannels(),
           "width": r.getsampwidth(), "frames": r.getnframes(),
           "seconds": r.getnframes()/r.getframerate(), "size": Path(out).stat().st_size})


In [ ]:
# Demo 2: codec / sample-rate decision helper
def recommend_audio(channel: str) -> dict:
    channel = channel.lower()
    if channel in {"pstn", "twilio", "phone"}:
        return {"rate": 8000, "codec": "mulaw/opus", "note": "narrowband telephony"}
    if channel in {"meetings", "zoom"}:
        return {"rate": 16000, "codec": "opus", "note": "speech-optimized"}
    if channel in {"music", "podcast"}:
        return {"rate": 48000, "codec": "aac/opus", "note": "full-band"}
    return {"rate": 16000, "codec": "pcm_s16le", "note": "default ASR-friendly"}
for ch in ("phone", "meetings", "podcast", "mobile_app"):
    print(ch, recommend_audio(ch))


## Audio Representations for ML

### Definition
Models consume waveforms, frames, spectrograms/mel features, or learned discrete audio tokens.

### Why it matters
Mismatched featurization (train 16 kHz, prod 8 kHz) silently destroys quality.

### How it works
Classic ASR: mel filterbanks/MFCCs → encoder. Modern: raw waveform / learned codecs end-to-end. TTS: spectrogram or codec tokens → vocoder.

### Intuition
Spectrograms are images of sound; LLMs increasingly eat audio tokens directly.

### Pitfalls
- Resampling with poor anti-alias filters
- Assuming stereo helps ASR (usually downmix)

### When to use
Any training/eval pipeline and production ingest normalization.


### Representations

| Form | Pros | Cons |
|------|------|------|
| PCM waveform | Lossless-ish fidelity | Large; model must learn filters |
| Mel spectrogram | Compact, speech-tuned | Lossy for some tasks |
| MFCCs | Tiny classic features | Weaker for deep E2E |
| Learned tokens | LLM-friendly | Codec artifacts / vendor lock |

```
time domain:  ~~~~waveform~~~~
freq domain:  ||||||  mel bands over time
tokens:       [a12][a77][a03] ...
```


In [ ]:
# Demo 3: frame energy VAD sketch (educational)
import math, struct, wave

def frame_energies(path, frame_ms=20):
    with wave.open(path) as r:
        rate, width, ch, n = r.getframerate(), r.getsampwidth(), r.getnchannels(), r.getnframes()
        assert width == 2 and ch == 1
        raw = r.readframes(n)
    samples = struct.unpack("<" + "h"*(len(raw)//2), raw)
    fsz = int(rate * frame_ms / 1000)
    energies = []
    for i in range(0, len(samples)-fsz, fsz):
        frame = samples[i:i+fsz]
        energies.append(sum(x*x for x in frame)/len(frame))
    return energies

write_sine_wav("demo_tone.wav", seconds=0.25)
E = frame_energies("demo_tone.wav")
thr = (sum(E)/len(E)) * 0.1
speechish = sum(1 for e in E if e > thr)
print("frames", len(E), "above_thr", speechish)


## System Constraints

### Definition
Voice UX is constrained by latency budgets, noise/accents, privacy/retention, and accessibility.

### Why it matters
A brilliant answer that arrives after 3 seconds of silence feels broken.

### How it works
Target ~200–500ms perceived responsiveness for conversational turns; stream partials; minimize cascade stalls; encrypt+TTL audio.

### Intuition
Phone calls punish you for every sequential round trip.

### Pitfalls
- Batch ASR in a duplex call
- Keeping raw audio forever 'just in case'
- No human transfer path

### When to use
Every production voice feature.


In [ ]:
# Demo 4: perceived latency decomposition
stages = {"vad_endpoint_ms": 180, "asr_final_ms": 220, "llm_ms": 450, "tts_ms": 200, "net_ms": 80}
print("sum", sum(stages.values()), "ms")
# speculative: start TTS on partial LLM tokens
stages2 = dict(stages); stages2["llm_ms"] = 180; stages2["tts_ms"] = 120
print("streamed", sum(stages2.values()), "ms")


In [ ]:
# Demo 5: privacy policy stub for audio
def retention_plan(product: str) -> dict:
    if product == "voice_agent":
        return {"audio_days": 0, "transcript_days": 30, "redact": ["ssn", "card"], "consent": True}
    if product == "call_qa":
        return {"audio_days": 30, "transcript_days": 365, "redact": ["ssn"], "consent": True}
    return {"audio_days": 7, "transcript_days": 30, "redact": [], "consent": True}
print(retention_plan("voice_agent"))
print(retention_plan("call_qa"))


### Accessibility & inclusion

| Concern | Practice |
|---------|----------|
| Deaf/HoH | Captions + visual confirmations |
| Speech disabilities | Longer endpointing; type fallback |
| Accents/dialects | Diverse eval sets; locale models |
| Cognitive load | Short TTS; confirm critical actions |


### Checklist — Speech fundamentals

- [ ] Sample rate/codec chosen per channel
- [ ] Latency budget written down
- [ ] Retention: audio vs transcript
- [ ] Accent/noise eval slice exists
- [ ] Human transfer path defined


### Try it yourself — Audio basics

1. Modify Demo 1 to write stereo and verify channels.
2. Resample mentally: what breaks if 8 kHz audio is scored as 16 kHz?
3. Write a 5-bullet privacy notice for a voice agent.

**Stretch:** Plot frame energies with matplotlib if available.


### Try it yourself — Constraints

1. Set an SLA for first audible TTS byte.
2. List 5 telephony-specific failure modes.


## Knowledge Check

**Q1.** Why is 8 kHz common on phone calls?

<details><summary>Answer</summary>

Legacy PSTN narrowband channels; many telephony APIs still deliver 8 kHz μ-law/opus.

</details>

**Q2.** Name three pillars besides ASR.

<details><summary>Answer</summary>

TTS, audio understanding, and voice agents (conversational systems).

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `ASR` | Automatic speech recognition |
| `TTS` | Text to speech |
| `VAD` | Voice activity detection |
| `sample rate` | Samples per second (Hz) |
| `codec` | Compression format for audio |
| `MOS` | Mean opinion score for perceived quality |


## Key Takeaways

- Speech AI = ASR + TTS + understanding + agents
- Normalize rate/codec before models
- Latency and privacy are first-class constraints
- Measure on noisy, accented, real-channel audio


## Production Incident Patterns — speech fundamentals

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "speech fundamentals",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — speech fundamentals

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("speech fundamentals", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — speech fundamentals ops

1. Draft an on-call runbook bullet list for speech fundamentals when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
